# Minesweeper Qwen GRPO on Kaggle T4x2

This version runs a short CPU-only smoke check. Add Kaggle Secrets named `HF_TOKEN` and (optionally) `HF_REPO_ID` for Hub checkpoint sync; without them, checkpoints remain in the notebook's local working directory. The launcher runs posterior SFT, truth/posterior GRPO, and evaluation.

In [ ]:
import os, subprocess
repo = '/kaggle/working/minesweeper-grpo'
if not os.path.isdir(os.path.join(repo, '.git')):
    subprocess.run(['git', 'clone', 'https://github.com/Anchal-T/minesweeper-grpo.git', repo], check=True)
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
try:
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
except Exception:
    print('HF_TOKEN not set; this run will keep checkpoints local')
try:
    os.environ['HF_REPO_ID'] = secrets.get_secret('HF_REPO_ID')
except Exception:
    print('HF_REPO_ID not set; checkpoints remain in Kaggle working storage')
python = os.environ.get('PY', __import__('sys').executable)
subprocess.run([python, '-m', 'pip', 'uninstall', '-y', 'torchao'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
subprocess.run([python, '-m', 'pip', 'install', '-q', 'peft', 'transformers', 'huggingface_hub'], check=True)
print('Repository ready:', repo)

In [ ]:
import os, subprocess, sys
os.chdir('/kaggle/working/minesweeper-grpo')
python = sys.executable
def run(args):
    subprocess.run([python, *args], check=True)
run(['sft.py', '--target', 'posterior', '--steps', '1', '--batch', '1', '--device', 'cpu', '--out', 'runs/cpu-sft', '--time-budget-min', '45'])
for reward in ('truth', 'posterior'):
    run(['grpo.py', '--init-from', 'runs/cpu-sft', '--reward', reward, '--adv-norm', 'none', '--kl-coef', '0.05', '--steps', '1', '--prompts-per-step', '1', '--group', '2', '--micro-batch', '2', '--device', 'cpu', '--out-dir', f'runs/cpu-{reward}', '--resume', '--time-budget-min', '45'])
for reward in ('truth', 'posterior'):
    run(['eval.py', '--ckpt', f'runs/cpu-{reward}/last', '--games', '1', '--device', 'cpu', '--batch-size', '1'])